In [39]:
!pip install scikit-learn==1.2.0
!pip install pandas as pd
!pip install joblib

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement as (from versions: none)
ERROR: No matching distribution found for as

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [40]:
import pandas as pd 
import numpy as np
import joblib


In [41]:
model = joblib.load("/Users/sadiqkhawaja/Desktop/PhD/CVD_RiskStrat_WebApp-merge-interface2/model_files/batch_models/TAVIModel.pkl")

In [42]:
exported_forest = []

for tree in model.estimators_:
    exported_forest.append({
        "feature": tree.tree_.feature.copy(),
        "threshold": tree.tree_.threshold.copy(),
        "left": tree.tree_.children_left.copy(),
        "right": tree.tree_.children_right.copy(),
        "value": tree.tree_.value[:,0,:].copy()
    })

In [43]:
forest_export = {
    "n_features_in": model.n_features_in_,
    "classes": model.classes_.tolist(),
    "trees": exported_forest
}

In [44]:
backendValues = {
    "Age":94,
    "Diabetes":0,
    "Ex_smoking":0,
    "Hypertension":1,
    "Creatinine":68,
    "Pre-operative_heart_rhythm":0,
    "Baseline_Hb":125,
    "Poor_mobility":1,
    "FEV1":1.6,
    "Predicted_VC":157,
    "FEV1_VC":0.727273,
    "Katz_Index":5,
    "TR":0
}

In [45]:
feature_order = [
    "Age",
    "Diabetes",
    "Ex_smoking",
    "Hypertension",
    "Creatinine",
    "Pre-operative_heart_rhythm",
    "Baseline_Hb",
    "Poor_mobility",
    "FEV1",
    "Predicted_VC",
    "FEV1_VC",
    "Katz_Index",
    "TR",
]

x = np.array([backendValues[f] for f in feature_order])

In [46]:
def predict_tree(tree, x):
    node = 0

    while tree["left"][node] != tree["right"][node]:
        feat = tree["feature"][node]
        thresh = tree["threshold"][node]

        if x[feat] <= thresh:
            node = tree["left"][node]
        else:
            node = tree["right"][node]

    return tree["value"][node]

In [47]:
def predict_forest_sklearn_style(forest_export, x):

    n_classes = len(forest_export["classes"])
    total = np.zeros(n_classes)

    for tree in forest_export["trees"]:

        leaf = predict_tree(tree, x).astype(float)

        # Convert this tree's leaf counts to probabilities
        tree_prob = leaf / leaf.sum()

        total += tree_prob

    return total / len(forest_export["trees"])

In [48]:
sk_prob = model.predict_proba([x])[0]
manual_prob = predict_forest_sklearn_style(forest_export, x)

print("sklearn :", sk_prob)
print("manual  :", manual_prob)
print("max diff:", np.max(np.abs(sk_prob - manual_prob)))
print("sklearn :", sk_prob)
print("manual  :", manual_prob)

print("max diff:", np.max(np.abs(sk_prob - manual_prob)))

sklearn : [0.64285714 0.35714286]
manual  : [0.64285714 0.35714286]
max diff: 0.0
sklearn : [0.64285714 0.35714286]
manual  : [0.64285714 0.35714286]
max diff: 0.0


In [49]:
print("Trees exported:", len(exported_forest))
print("Model trees:", len(model.estimators_))

Trees exported: 154
Model trees: 154


In [50]:
dir(model)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_check_feature_names',
 '_check_n_features',
 '_compute_oob_predictions',
 '_estimator',
 '_estimator_type',
 '_get_oob_predictions',
 '_get_param_names',
 '_get_tags',
 '_make_estimator',
 '_more_tags',
 '_parameter_constraints',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_required_parameters',
 '_set_oob_score_and_attributes',
 '_validate_X_predict',
 '_validate_data',
 '_validate_estimator',
 '_validate_params',
 '_validate_y_class_weight',
 'apply',
 'base_estimator',


In [51]:
len(model.feature_importances_)

13

In [54]:
exported_forest = []

for tree in model.estimators_:
    exported_forest.append({
        "feature": tree.tree_.feature.tolist(),
        "threshold": tree.tree_.threshold.tolist(),
        "left": tree.tree_.children_left.tolist(),
        "right": tree.tree_.children_right.tolist(),
        "value": tree.tree_.value[:,0,:].tolist()
    })

forest_export = {
    "n_features_in": int(model.n_features_in_),
    "classes": model.classes_.tolist(),
    "trees": exported_forest
}

In [55]:
import json

with open("forest_export.json", "w") as f:
    json.dump(forest_export, f)